# Data Quality Correction

Before performing EDA (Exploratory Data Analysis), some initial data cleaning is necessary so you can trust what you’re analyzing.

**Basic cleaning includes:**

- Loading the data correctly (e.g., proper parsing of dates, encoding, etc.)

- Checking for and handling:

    - Missing values (at least identifying them)

    - Incorrect data types (e.g., string instead of numeric)

    - Obvious outliers or garbage values (e.g., negative age)

    - Duplicate rows

> 🧼 This stage is often called "initial or light cleaning" — it helps avoid misleading results during EDA.

**See Run Results :** [https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow](https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow/#/)

## Required Installtion

In [ ]:
# TRIGGER_FLAG: run
import os
import sys

# Check if running in Kaggle Cloud Environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    print("--- Kaggle Environment Detected: Installing Cloud Dependencies ---")
    # 1. Install general cloud dependencies
    os.system(f"{sys.executable} -m pip install -q mlflow dagshub")
    
    # 2. Clone the specific branch of the repository
    print("Cloning repository...")
    os.system("git clone -b feat/notebooks https://github.com/Rahul-Shelke-1/heart-stroke-risk-stratification.git")
    
    # 3. Change directory into the cloned repo and install it in editable mode
    if os.path.exists("heart-stroke-risk-stratification"):
        os.chdir("heart-stroke-risk-stratification")
        print("Installing package in editable mode...")
        os.system(f"{sys.executable} -m pip install -e .")
        # Optional: Change directory back to the original root if needed
        # os.chdir("..")
    else:
        print("Error: Repository directory not found.")
        
else:
    print("--- Local/Non-Kaggle Environment Detected: Skipping Installation ---")
    # Local packages should ideally be pre-managed via your local uv environment

--- Local/Non-Kaggle Environment Detected: Skipping Installation ---


## Smart Environment Detection & Secrets Setup

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

# 1. Safely retrieve credentials from Kaggle's internal secure vault
try:
    # 1. Detect if running inside Kaggle's container
    IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

    if IS_KAGGLE:
        # Import dynamically so local runs do not crash on ModuleNotFoundError
        from kaggle_secrets import UserSecretsClient

        user_secrets = UserSecretsClient()
        DAGSHUB_USERNAME = user_secrets.get_secret("DAGSHUB_USERNAME")
        DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")
    else:
        load_dotenv()
        DAGSHUB_USERNAME = os.environ.get("DAGSHUB_USERNAME", "Rahul-404")
        DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN")
    
    # Replace this with your actual DagsHub repo name
    REPO_NAME = "heart-stroke-prediction"
    
    # 2. Inject environment variables that the MLflow client natively looks for
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    
    # 3. Set the remote tracking URI to point to DagsHub
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
    
    print("Successfully connected to DagsHub MLflow tracking server!")
except Exception as e:
    # print(f"Local or non-Kaggle execution environment detected: {e}")
    print(f"Error establishing MLflow/DagsHub context: {e}")

/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully connected to DagsHub MLflow tracking server!


## Unified Path Setup Strategy

In [ ]:
import os
from pathlib import Path

# 1. Reuse your environment checker flag
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# 2. Define your repository name to map Kaggle input paths perfectly
KAGGLE_USER = user_secrets.get_secret("KAGGLE_USER")  # Replace with your actual lowercase Kaggle username
DATASET_SLUG = user_secrets.get_secret("KAGGLE_DATASET_NAME")  # Replace with your Kaggle dataset slug

if IS_KAGGLE:
    # --- KAGGLE CLOUD PATH CONTEXT ---
    # Kaggle mounts your datasets under /kaggle/input/YOUR_DATASET_SLUG
    INPUT_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USER}/{DATASET_SLUG}")

    if not INPUT_DIR.exists():
        raise f"Input Directory not sxists: {INPUT_DIR}"
    
    # Kaggle strictly allows file writing ONLY inside /kaggle/working
    OUTPUT_DIR = Path("/kaggle/working/artifacts")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    if not OUTPUT_DIR.exists():
        raise f"Output Directory not sxists: {OUTPUT_DIR}"
else:
    # --- LOCAL ENVIRONMENT PATH CONTEXT ---
    current_directory = Path.cwd()
    # Points to your local repository structure (e.g., repository_root/data/)
    INPUT_DIR = Path(current_directory, "notebooks", "data")
    
    if not INPUT_DIR.exists():
        raise f"Input Directory not sxists: {INPUT_DIR}"

    # Keeps your local repo clean by saving local run artifacts into an outputs folder
    OUTPUT_DIR = Path(current_directory, "notebooks", "artifact")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("--- PATH SUMMARY ---")
print(f"Execution Target: {'Kaggle Cloud' if IS_KAGGLE else 'Local Machine'}")
print(f"Reading Input From: {INPUT_DIR}")
print(f"Writing Outputs To: {OUTPUT_DIR}")

--- PATH SUMMARY ---
Execution Target: Local Machine
Reading Input From: /Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/notebooks/data
Writing Outputs To: /Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/notebooks/artifact


## Importing Libraries

In [68]:
from src.analyze.data_correction.schema_correction import (MissingValueNormalizer, FrequencyFilter)
from src.analyze.data_correction.schema_correction import (ColumnNameCorrector, ValueCorrector)
from src.analyze.data_inspection.inspector import InspectData
from sklearn.pipeline import Pipeline
import mlflow.sklearn
import pandas as pd
import cloudpickle
import warnings
import mlflow
import os

warnings.filterwarnings("ignore")

In [4]:
DATA_FILES = os.listdir(INPUT_DIR)
DATA_FILES

['healthcare-dataset-stroke-data.csv']

In [5]:
ARTIFACT_FILES = os.listdir(OUTPUT_DIR)
ARTIFACT_FILES

['data_correction_pipeline.pkl']

In [6]:
DATA_FILE_NAME = {
    'raw': Path(INPUT_DIR, 'healthcare-dataset-stroke-data.csv'),
    'clean': Path(INPUT_DIR, 'heart_stroke_data.csv'), 
}

ARTIFACT_FILE_NAME = {
    'data_correct': Path(OUTPUT_DIR, 'data_correction_pipeline.pkl'),
}

### Set Experiment Name

In [7]:
experiment_name = "01_Data_Quality_Correction"
experiment_tags = {
    "experiment_type": "Classification",
    "project": "heart_stroke_risk_stratification",
    "team": "Data_Science_Core"
}

try:
    experiment_id = mlflow.create_experiment(name=experiment_name, tags=experiment_tags)
except Exception:
    # If it already exists, fetch its ID
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# 3. Set it as the active experiment
mlflow.set_experiment(experiment_id=experiment_id)

<Experiment: artifact_location='mlflow-artifacts:/c0b296281c454824950088372a9da4dd', creation_time=1783515482449, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783515482449, lifecycle_stage='active', name='01_Data_Quality_Correction', tags={'experiment_type': 'Classification',
 'mlflow.experimentKind': 'custom_model_development',
 'project': 'heart_stroke_risk_stratification',
 'team': 'Data_Science_Core'}, trace_location=None, workspace='default'>

### Set Run Name

In [8]:
# Starts a global active run
mlflow.start_run(run_name="Data_Quality_Correction_Session")

<ActiveRun: >

## Import Data

In [9]:
df = pd.read_csv(DATA_FILE_NAME['raw'])
dataset = mlflow.data.from_pandas(df, name="raw_data", targets="stroke")
mlflow.log_input(dataset, context="dataset_ingestion")
mlflow.log_param("raw_data_path",DATA_FILE_NAME['raw'])

PosixPath('/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/notebooks/data/healthcare-dataset-stroke-data.csv')

In [10]:
mlflow.log_param("num_train_samples", df.shape[0])
mlflow.log_param("num_features", df.shape[1])
df.shape

(5110, 12)

In [11]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [12]:
df['stroke'].value_counts()

stroke
0    4861
1     249
Name: count, dtype: int64

In [13]:
stroke_proportions = df['stroke'].value_counts(normalize=True).to_dict()

mlflow.log_param("target_class_0_prop", stroke_proportions[0])
mlflow.log_param("target_class_1_prop", stroke_proportions[1])

stroke_proportions

{0: 0.9512720156555773, 1: 0.0487279843444227}

In [14]:
df['stroke'].value_counts(normalize=True)

stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64

## 1. Check Schema, shape, columns

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


- **~ 5k rows**
- **12 columns** (including target column)

## 2. Standardize Column Names

In [16]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

- will keep all to lower case.

In [17]:
column_correction_dict = {
    'Residence_type': 'residence_type'
}

# Save the dictionary directly to MLflow as a JSON file
mlflow.log_dict(column_correction_dict, "artifact/preprocessing/column_mapping.json")

In [18]:
column_corrector = ColumnNameCorrector(column_correction_dict)

In [19]:
df = column_corrector.transform(df)

In [20]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. Fix Categorical Values (String Standardization)

In [21]:
inspector = InspectData(df)

In [22]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

In [23]:
cat_col = ['gender', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'smoking_status', 'stroke']

In [24]:
inspector.categorical_column_inspect(cat_col)

-------------------------

column name: gender

# unique: 3

unique values: ['Male' 'Female' 'Other']

max: Female

min: Other

null count: 0

-------------------------

column name: hypertension

# unique: 2

unique values: [0 1]

max: 0

min: 1

null count: 0

-------------------------

column name: heart_disease

# unique: 2

unique values: [1 0]

max: 0

min: 1

null count: 0

-------------------------

column name: ever_married

# unique: 2

unique values: ['Yes' 'No']

max: Yes

min: No

null count: 0

-------------------------

column name: work_type

# unique: 5

unique values: ['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']

max: Private

min: Never_worked

null count: 0

-------------------------

column name: residence_type

# unique: 2

unique values: ['Urban' 'Rural']

max: Urban

min: Rural

null count: 0

-------------------------

column name: smoking_status

# unique: 4

unique values: ['formerly smoked' 'never smoked' 'smokes' 'Unknown']

max: never s

**Correction needed to this columns:**
- gender
- ever_married
- work_type
- residence_type
- smoking_status

In [25]:
value_correction_dict = {
    'gender': {'Male': 'male', 'Female': 'female', 'Other': 'other'},
    'ever_married': {'Yes':'yes', 'No': 'no'},
    'residence_type': {'Urban': 'urban', 'Rural': 'rural'},
    'smoking_status': {'formerly smoked': 'formerly_smoked', 
                            'never smoked': 'never_smoked', 
                            'Unknown': 'unknown'},
    'work_type': {'Private':'private', 'self-employed': 'self_employed', 
                            'Govt_job': 'govt_job', 'Never_worked': 'never_worked',
                            'children': 'children'},
}

mlflow.log_dict(value_correction_dict, "artifact/preprocessing/value_mapping.json")

In [26]:
value_corrector = ValueCorrector(column_mappings=value_correction_dict)

In [27]:
df = value_corrector.transform(X=df)

In [28]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [29]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,unknown,0


## 4. Fix Categorical Values Frequency (Handling category frequency)

In [30]:
frequency_filter = FrequencyFilter(threshold=1, action='drop')

# Log the initialization parameters
mlflow.log_param("freq_filter_threshold", frequency_filter.threshold)
mlflow.log_param("freq_filter_action", frequency_filter.action)

'drop'

In [31]:
df = frequency_filter.fit_transform(df)

In [32]:
df.shape

(5109, 12)

In [33]:
df[df['gender'] == 'other']

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


In [34]:
df['gender'].value_counts()

gender
female    2994
male      2115
Name: count, dtype: int64

In [35]:
# Assuming your class stores the dropped labels in an attribute after fitting
if hasattr(frequency_filter, "fill_value"):
    mlflow.log_dict(
        {"dropped_labels": [frequency_filter.fill_value]},
        "artifact/preprocessing/dropped_rare_categories.json",
    )

## 5. Fix Numerical Values (Context Standardization)

In [36]:
num_col = ['age', 'avg_glucose_level', 'bmi']

In [37]:
inspector.numerical_column_inspect(num_col)

-------------------------

column name: age

# unique: 104

+ve values: True | (104)

zeros: 0

-ve values: False | (0)

null values: 0

-------------------------

column name: avg_glucose_level

# unique: 3979

+ve values: True | (3979)

zeros: 0

-ve values: False | (0)

null values: 0

-------------------------

column name: bmi

# unique: 418

+ve values: True | (418)

zeros: 0

-ve values: False | (0)

null values: 201

-------------------------


**bmi**

- `bmi` feature has 4% (201) values missing
<!-- - replacing `nan` with -1 -->

⚠️ **Unusual / Suspicious Values:**

| Range                               | Possible Issue                                                   |
| ----------------------------------- | ---------------------------------------------------------------- |
| **< 40**                            | Rare, possible hypoglycemia or data error                        |
| **> 250–300**                       | Very high, may indicate poorly controlled diabetes or unit error |
| **> 600**                           | Extremely rare, possibly a data or entry error                   |
| **Unrealistic values (e.g. >1000)** | Almost certainly data error                                      |


## 6. Remove Unwanted/Corrupted Columns/Rows

In [38]:
df[df.duplicated()]

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


In [39]:
df[df.drop(columns="id", axis=1).duplicated()]

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


## 7. Identifying missing values (off context)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5109 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5109 non-null   int64  
 1   gender             5109 non-null   object 
 2   age                5109 non-null   float64
 3   hypertension       5109 non-null   int64  
 4   heart_disease      5109 non-null   int64  
 5   ever_married       5109 non-null   object 
 6   work_type          5109 non-null   object 
 7   residence_type     5109 non-null   object 
 8   avg_glucose_level  5109 non-null   float64
 9   bmi                4908 non-null   float64
 10  smoking_status     5109 non-null   object 
 11  stroke             5109 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 518.9+ KB


In [41]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [42]:
df[df['smoking_status'] == 'unknown']

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
8,27419,female,59.0,0,0,yes,private,rural,76.15,NaN,unknown,1
9,60491,female,78.0,0,0,yes,private,urban,58.57,24.2,unknown,1
13,8213,male,78.0,0,1,yes,private,urban,219.84,NaN,unknown,1
19,25226,male,57.0,0,1,no,govt_job,urban,217.08,NaN,unknown,1
23,64778,male,82.0,0,1,yes,private,rural,208.30,32.5,unknown,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5098,579,male,9.0,0,0,no,children,urban,71.88,17.5,unknown,0
5101,36901,female,45.0,0,0,yes,private,urban,97.95,24.5,unknown,0
5103,22127,female,18.0,0,0,no,private,urban,82.85,46.9,unknown,0
5104,14180,female,13.0,0,0,no,children,rural,103.08,18.6,unknown,0


In [43]:
missing_normalizer = MissingValueNormalizer(missing_tokens=['unknown'])

In [44]:
df = missing_normalizer.fit_transform(df)

In [45]:
df.isna().sum()

id                      0
gender                  0
age                     0
hypertension            0
heart_disease           0
ever_married            0
work_type               0
residence_type          0
avg_glucose_level       0
bmi                   201
smoking_status       1544
stroke                  0
dtype: int64

In [46]:
missing_series = df.isnull().sum()
missing_params = {f"post_norm_missing_{col}": int(val) for col, val in missing_series.items()}

In [47]:
# Log the normalizer configurations
mlflow.log_param("missing_normalizer_tokens", str(missing_normalizer.missing_tokens))

# Log all column missing counts simultaneously using a single dictionary batch
mlflow.log_params(missing_params)

## **Pipeline: Basic Cleaning**

### 1. Build pipeline

In [48]:
# Create pipeline
data_correction_pipeline = Pipeline(
    steps=[
        ('correct_column_names', ColumnNameCorrector(mapping=column_correction_dict)),
        ('correct_column_values', ValueCorrector(column_mappings=value_correction_dict)),
        ('missing_normalizer', MissingValueNormalizer(missing_tokens=['unknown'])),
        ('frequency_filter', FrequencyFilter(threshold=1, action='drop')),
    ]
)

In [49]:
data_correction_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('correct_column_names', ...), ('correct_column_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapping,{'Residence_type': 'residence_type'}
,column_mappings,"{'ever_married': {'No': 'no', 'Yes': 'yes'}, 'gender': {'Female': 'female', 'Male': 'male', 'Other': 'other'}, 'residence_type': {'Rural': 'rural', 'Urban': 'urban'}, 'smoking_status': {'Unknown': 'unknown', 'formerly smoked': 'formerly_smoked', 'never smoked': 'never_smoked'}, ...}"
,missing_tokens,['unknown']
,threshold,1
,action,'drop'
,fill_value,'other'


#### data-head

In [50]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [51]:
data_correction_pipeline.fit_transform(df).head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


#### data-tail

In [52]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,NaN,0


In [53]:
data_correction_pipeline.fit_transform(df).tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,nan,0


### 2. Apply pipeline

In [54]:
df = pd.read_csv(DATA_FILE_NAME['raw'])

# df.head()

In [55]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [56]:
df_corrected = data_correction_pipeline.fit_transform(df)

In [57]:
df_corrected.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [58]:
df_corrected.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,NaN,0


In [59]:
df_corrected.isna().sum()

id                      0
gender                  0
age                     0
hypertension            0
heart_disease           0
ever_married            0
work_type               0
residence_type          0
avg_glucose_level       0
bmi                   201
smoking_status       1544
stroke                  0
dtype: int64

### 3. Save pipeline & Data

In [60]:
local_file_path = ARTIFACT_FILE_NAME["data_correct"]

with open(local_file_path, "wb" ) as f:
    cloudpickle.dump(data_correction_pipeline, f)

In [61]:
with open(local_file_path, "rb" ) as f:
    data_correction_pipeline = cloudpickle.load(f)

In [62]:
data_correction_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('correct_column_names', ...), ('correct_column_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapping,{'Residence_type': 'residence_type'}
,column_mappings,"{'ever_married': {'No': 'no', 'Yes': 'yes'}, 'gender': {'Female': 'female', 'Male': 'male', 'Other': 'other'}, 'residence_type': {'Rural': 'rural', 'Urban': 'urban'}, 'smoking_status': {'Unknown': 'unknown', 'formerly smoked': 'formerly_smoked', 'never smoked': 'never_smoked'}, ...}"
,missing_tokens,['unknown']
,threshold,1
,action,'drop'
,fill_value,'other'


In [ ]:
mlflow.sklearn.log_artifact(local_file_path, artifact_path="artifact/pipelines/data_cleaning")

# Optional: Log a metadata tag identifying it as a static pipeline
mlflow.set_tag("pipeline.type", "static_etl")

In [64]:
# Make sure to close the run so MLflow knows the notebook is done
mlflow.end_run()

🏃 View run Data_Quality_Correction_Session at: https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow/#/experiments/1/runs/737370d1d11c49499fe4cffdfe6c94c3
🧪 View experiment at: https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow/#/experiments/1


### **Next Action:**

- Handle Missing Values
- Handle Outliers